In [ ]:
# --- 导入所有必要的库 ---
# 数据处理: numpy, pandas, Path (文件路径管理)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json

# 机器学习模型: RandomForestRegressor (随机森林回归)
# 评估指标: MAE(平均绝对误差), MSE(均方误差), R²(决定系数)
# 交叉验证: KFold (K折交叉验证), train_test_split (训练/验证集划分)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split

# MatBench: 材料科学基准数据集加载工具
# MatMiner: 化学成分和结构特征提取器
from matbench.bench import MatbenchBenchmark
from matminer.featurizers.composition import ElementProperty
from matminer.featurizers.base import MultipleFeaturizer

# XGBoost: 梯度提升决策树库
import xgboost as xgb
import shap

print("✅ 所有库导入成功!")

In [ ]:
# ==================== 配置参数 ====================
# 模型配置
MODELS_TO_RUN = ["rf", "xgb"]  # 运行的模型列表: RF(随机森林) 和 XGB(XGBoost)
N_JOBS = 8  # 并行处理的CPU核心数量

# 任务和路径
TASK_NAME = "matbench_mp_gap"  # MatBench任务名称: 预测材料的带隙能
OUT_DIR = Path("../outputs_v1_run0709")  # 输出目录路径 (本次运行的冻结版本; 新目录=强制重新提取特征)
OUT_DIR.mkdir(parents=True, exist_ok=True)  # 创建输出目录（递归创建父目录）

# 缓存和复用设置
FEATURES_CACHE = OUT_DIR / "features_cache.pkl"  # 特征缓存文件路径
FORCE_RETRAIN = False  # 是否强制重新训练 (True=重新生成, False=复用旧预测)

# 模型超参数
SEED = 42  # 随机种子 (保证可重复性)
N_SPLITS = 5  # OOF交叉验证的折数
TOPK = 50  # 导出错误最大的前50个样本

# ==================== 加载MatBench任务 ====================
# MatBenchBenchmark 是材料科学的标准基准
# autoload=True 表示自动加载所有可用的任务
mb = MatbenchBenchmark(autoload=True)

# 从所有任务中找到指定任务 (matbench_mp_gap 是预测MP数据库中材料的带隙)
task = next(t for t in mb.tasks if t.dataset_name == TASK_NAME)
task.load()  # 加载任务数据

# 获取目标列名 (通常为 'gap pbe' 或类似的能隙值列)
target_col = task.metadata["target"]
print(f"✅ 任务加载成功: {TASK_NAME}")

In [ ]:
# ==================== 特征提取函数 ====================
def featurize_structures(structures):
    """
    从结构列表中提取化学成分特征
    
    参数:
        structures (list): 结构对象列表 (来自MatBench)
    
    返回:
        pd.DataFrame: 提取的特征矩阵 (样本数 × 特征数)
        
    功能步骤:
    1. 从结构中提取化学成分信息
    2. 使用 Magpie 预设特征器进行特征提取 (包括原子物理性质等)
    3. 清理无穷大和NaN值，用平均值填充
    """
    if not structures:
        return pd.DataFrame()
    
    # 提取每个结构的化学成分
    compositions = [s.composition for s in structures]
    tmp_df = pd.DataFrame({"comp": compositions})
    
    # 使用Magpie预设特征器 (包含元素周期表性质如原子半径、电负性等)
    featurizers = [ElementProperty.from_preset("magpie")]
    featurizer = MultipleFeaturizer(featurizers)
    featurizer.set_n_jobs(N_JOBS)  # 并行处理
    featurizer.set_chunksize(100)  # 每批处理100个样本
    
    print(f"  正在提取特征 (使用 {N_JOBS} 个CPU核心)...")
    featured_df = featurizer.featurize_dataframe(tmp_df, "comp")
    
    # 数据清理: 将无穷大替换为NaN，用列的平均值填充NaN
    featured_df = featured_df.replace([np.inf, -np.inf], np.nan)  # NaN 保留，填充延后到 fold 内执行 (fit on train)
    
    # 删除成分列，只保留特征
    X = featured_df.drop(columns=["comp"], errors="ignore")
    print(f"  ✅ 特征提取完成! 特征数量: {X.shape[1]}")
    return X


# ==================== 模型创建函数 ====================
def make_model(model_name: str):
    """
    根据模型名称创建相应的机器学习模型
    
    参数:
        model_name (str): 模型名称 ('rf' 或 'xgb')
    
    返回:
        RandomForestRegressor 或 None (XGBoost在其他函数中单独处理)
    """
    if model_name == "rf":
        print("  使用模型: 随机森林 (Random Forest)")
        # n_estimators: 树的数量,random_state: 随机种子, n_jobs: 使用所有CPU核心
        return RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)
    elif model_name == "xgb":
        print("  使用模型: XGBoost (XGBRegressor)")
        return None  # XGBoost 在 xgb_predict_with_early_stop 函数中单独处理
    else:
        raise ValueError(f"❌ Unknown model: {model_name}")

In [ ]:
# ==================== XGBoost模型训练和预测 ====================
def xgb_predict_with_early_stop(X_train, y_train, X_test, fold_idx: int, seed=SEED,
                                num_boost_round=8000, early_stopping_rounds=200,
                                verbose_eval=50):
    """
    使用XGBoost训练模型，支持早停和损失曲线记录
    
    参数:
        X_train, y_train: 训练集特征和目标值
        X_test: 测试集特征
        fold_idx: 当前折数索引 (用于日志输出)
        seed: 随机种子
        num_boost_round: 最大提升轮数
        early_stopping_rounds: 早停轮数 (验证集性能不改进则停止)
        verbose_eval: 每多少轮打印一次评估结果
    
    返回:
        y_pred: 测试集的预测值
        
    功能步骤:
    1. 将训练集分为内部训练集和验证集 (用于early stopping)
    2. 训练XGBoost模型
    3. 保存验证MAE损失曲线图
    4. 保存训练信息到JSON文件
    5. 预测并返回测试集结果
    """
    
    # 内部训练/验证集划分 (80%训练, 20%验证)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=seed, shuffle=True
    )

    # 将数据转换为XGBoost的DMatrix格式
    dtr = xgb.DMatrix(X_tr, label=y_tr)  # 训练集
    dval = xgb.DMatrix(X_val, label=y_val)  # 验证集
    dtest = xgb.DMatrix(X_test)  # 测试集

    # XGBoost超参数配置
    params = {
        "objective": "reg:squarederror",  # 回归任务，使用平方误差损失
        "eval_metric": "mae",  # 评估指标: 平均绝对误差
        "eta": 0.03,  # 学习率 (越小越稳定，需要更多轮数)
        "max_depth": 6,  # 树的最大深度
        "min_child_weight": 5,  # 叶子节点最小样本权重
        "subsample": 0.8,  # 行采样比例 (每棵树使用80%的样本)
        "colsample_bytree": 0.8,  # 列采样比例 (每棵树使用80%的特征)
        "lambda": 1.0,  # L2正则化参数
        "alpha": 1e-3,  # L1正则化参数
        "tree_method": "hist",  # 使用直方图方法加速训练
        "seed": seed,
    }

    # 记录评估结果 (用于early stopping和绘图)
    evals_result = {}

    # 训练XGBoost模型
    booster = xgb.train(
        params=params,
        dtrain=dtr,
        num_boost_round=num_boost_round,
        evals=[(dval, "validation")],  # 在验证集上评估
        evals_result=evals_result,
        early_stopping_rounds=early_stopping_rounds,  # 如果验证性能不改进则停止
        verbose_eval=verbose_eval  # 每50轮打印一次结果
    )

    # 获取最佳迭代次数和分数
    best_it = booster.best_iteration
    best_score = booster.best_score

    print(f"[XGB Fold {fold_idx+1}] best_iteration = {best_it}, best_mae = {best_score:.6f}")

    # 使用最佳模型预测测试集
    try:
        y_pred = booster.predict(dtest, iteration_range=(0, best_it + 1))
    except TypeError:
        # 兼容旧版XGBoost API
        y_pred = booster.predict(dtest, ntree_limit=best_it + 1)

    # ==================== 保存训练曲线和信息 ====================
    if 'validation' in evals_result and 'mae' in evals_result['validation']:
        # 提取验证集MAE数据
        val_mae_list = evals_result['validation']['mae']
        epochs = list(range(1, len(val_mae_list) + 1))

        # 绘制MAE损失曲线
        plt.figure(figsize=(10, 6))
        plt.plot(epochs, val_mae_list, label='Validation MAE', color='blue')
        # 画一条竖线标记最佳迭代点
        plt.axvline(x=best_it + 1, color='red', linestyle='--',
                    label=f'Best: {best_it} (MAE={best_score:.6f})')
        plt.xlabel('Boosting Rounds')
        plt.ylabel('Validation MAE (eV)')
        plt.title(f'XGBoost Validation MAE Curve - Fold {fold_idx+1}')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # 保存曲线图
        curve_path = OUT_DIR / f"xgb_mae_curve_fold_{fold_idx+1}.png"
        plt.savefig(curve_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  → MAE 曲线保存: {curve_path}")

        # 保存训练信息到JSON
        info = {
            "fold": fold_idx + 1,
            "best_iteration": int(best_it),  # 最佳迭代次数
            "best_validation_mae": float(best_score),  # 最佳验证MAE
            "early_stopping_rounds": early_stopping_rounds,
            "total_rounds_evaluated": len(val_mae_list),  # 实际评估的轮数
            "final_val_mae": float(val_mae_list[-1]),  # 最后一轮的验证MAE
            "params": params  # 模型超参数配置
        }
        info_path = OUT_DIR / f"xgb_training_info_fold_{fold_idx+1}.json"
        with open(info_path, 'w', encoding='utf-8') as f:
            json.dump(info, f, indent=2, ensure_ascii=False)
        print(f"  → 训练信息保存: {info_path}")

    return booster, y_pred

In [ ]:
# ==================== SHAP 分析函数 (与 v2 相同的修正版) ====================
def run_shap_analysis(booster_or_model, X_train, feature_names, fold_idx: int, model_name: str):
    """计算 SHAP 值（随机抽样 3000），保存蜂群图、重要性条形图、原始值和特征名"""
    print(f'\n  [SHAP 分析] fold {fold_idx}...')

    n_shap = min(3000, len(X_train))
    rng = np.random.RandomState(SEED)
    idx = rng.choice(len(X_train), n_shap, replace=False)
    X_shap = X_train.iloc[idx] if hasattr(X_train, 'iloc') else X_train[idx]

    explainer = shap.TreeExplainer(booster_or_model)
    shap_values = explainer.shap_values(X_shap, check_additivity=False)

    # 蜂群图
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_shap, feature_names=feature_names, show=False, max_display=20)
    plt.title(f'SHAP beeswarm - {model_name} fold {fold_idx}')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'shap_beeswarm_{model_name}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
    plt.close()

    # 重要性条形图
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_shap, feature_names=feature_names,
                      plot_type='bar', show=False, max_display=20)
    plt.title(f'SHAP importance - {model_name} fold {fold_idx}')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'shap_importance_{model_name}_fold_{fold_idx}.png', dpi=150, bbox_inches='tight')
    plt.close()

    # 原始 SHAP 值 + 特征名
    np.save(OUT_DIR / f'shap_values_{model_name}_fold_{fold_idx}.npy', shap_values)
    with open(OUT_DIR / f'shap_feature_names_{model_name}.json', 'w', encoding='utf-8') as f:
        json.dump(list(feature_names), f, ensure_ascii=False)
    print(f'  → SHAP 输出已保存 (fold {fold_idx})')

In [ ]:
# ==================== 运行单个模型的主函数 ====================
def run_one_model(model_name: str):
    """
    完整训练流程: 遍历所有fold进行交叉验证，保存预测结果，计算最终分数
    
    参数:
        model_name (str): 模型名称 ('rf' 或 'xgb')
    
    返回:
        分数字典 (从MatBench官方验证返回)
        
    功能步骤:
    1. 创建预测结果保存目录
    2. 加载或创建特征缓存
    3. 对每个fold:
       - 读取训练和测试数据
       - 提取特征（或从缓存加载）
       - 训练模型并保存预测
       - 向MatBench任务记录预测结果
    4. 保存特征缓存
    5. 获取官方分数并保存
    """
    
    # 创建预测结果保存目录
    predictions_dir = OUT_DIR / f"predictions_{model_name}"
    predictions_dir.mkdir(parents=True, exist_ok=True)
    
    # 尝试加载特征缓存（如果之前已提取过）
    all_features = {} if not FEATURES_CACHE.exists() else pd.read_pickle(FEATURES_CACHE)
    features_cache_exists = FEATURES_CACHE.exists()

    # 创建新的task对象用于本模型评估
    # 每个模型需要独立的task对象来记录预测
    mb_local = MatbenchBenchmark(autoload=True)
    task_local = next(t for t in mb_local.tasks if t.dataset_name == TASK_NAME)
    task_local.load()

    # ==================== 遍历每个fold ====================
    for fold_idx, fold in enumerate(task_local.folds):
        print(f"\n[{model_name}] Fold {fold_idx+1}/{len(task_local.folds)}")
        
        # 预测文件路径
        pred_file = predictions_dir / f"pred_fold_{fold}.npy"
        
        # 如果预测文件已存在且无需重新训练，则直接加载并跳过
        if pred_file.exists() and not FORCE_RETRAIN:
            y_pred = np.load(pred_file)
            task_local.record(fold, y_pred)  # 向task记录预测
            continue

        # ========== 数据加载和清理 ==========
        # 获取训练和验证数据
        train_df = task_local.get_train_and_val_data(fold, as_type="df")
        test_df = task_local.get_test_data(fold, as_type="df")
        
        # MatBench数据可能包含空格，需要清理列名
        train_df = train_df.rename(columns=lambda x: x.strip())
        test_df = test_df.rename(columns=lambda x: x.strip())
        
        # 提取输入特征（结构）和目标值（能隙）
        input_col = "structure"
        X_train_raw = train_df[input_col].tolist()  # 原始结构对象
        y_train = train_df[target_col].to_numpy()  # 目标值（带隙）
        X_test_raw = test_df[input_col].tolist()  # 测试集结构对象

        # ========== 特征提取（或从缓存加载） ==========
        cache_key = f"fold_{fold}"
        
        if features_cache_exists:
            # 从缓存加载特征
            X_train = all_features[f"{cache_key}_train"]
            X_test = all_features[f"{cache_key}_test"]
        else:
            # 提取特征
            X_train = featurize_structures(X_train_raw)
            X_test = featurize_structures(X_test_raw)
            # 保存到缓存字典
            all_features[f"{cache_key}_train"] = X_train
            all_features[f"{cache_key}_test"] = X_test

        # ========== 填充: 统计量只从训练集来 (fit on train, transform on test) ==========
        col_means = X_train.mean(numeric_only=True)
        X_train = X_train.fillna(col_means).fillna(0)
        X_test = X_test.fillna(col_means).fillna(0)

        # ========== 模型训练和预测 ==========
        if model_name == "xgb":
            # XGBoost: 使用带early stopping的函数
            booster, y_pred = xgb_predict_with_early_stop(
                X_train, y_train, X_test,
                fold_idx=fold_idx,
            )
            run_shap_analysis(booster, X_train, list(X_train.columns), fold_idx, model_name)
        else:
            # Random Forest: 直接训练和预测
            model = make_model(model_name)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            if fold_idx == 0:   # RF 的 SHAP 只跑 fold 0
                run_shap_analysis(model, X_train, list(X_train.columns), fold_idx, model_name)

        # ========== 保存预测结果 ==========
        y_pred = np.maximum(y_pred, 0)  # 物理约束: 带隙非负
        np.save(pred_file, y_pred)  # 保存为numpy文件
        task_local.record(fold, y_pred)  # 向MatBench task记录预测

    # ========== 保存特征缓存 ==========
    # 仅在首次提取特征时保存缓存
    if not features_cache_exists and all_features:
        pd.to_pickle(all_features, FEATURES_CACHE)
        print(f"✅ 特征缓存已保存: {FEATURES_CACHE}")

    # ========== 官方分数计算 ==========
    # MatBench会自动计算标准测试集上的分数
    scores = task_local.validate() or task_local.scores
    
    # 保存分数到文本文件
    metrics_path = OUT_DIR / f"metrics_{model_name}.txt"
    with metrics_path.open("w") as f:
        f.write(f"任务: {TASK_NAME}\n模型: {model_name}\n分数:\n{str(scores)}\n")
    print(f"✅ 分数保存: {metrics_path}")
    
    return scores

In [ ]:
# ==================== 主流程: 训练模型并在官方测试集上评分 ====================
# 这一部分对MODELS_TO_RUN列表中的每个模型进行完整训练流程
# 包括5-fold交叉验证，预测保存，以及官方分数计算

all_scores = {}  # 存储所有模型的最终分数

# 遍历每个模型
for model_name in MODELS_TO_RUN:
    print(f"\n=== 跑模型: {model_name} ===")
    all_scores[model_name] = run_one_model(model_name)

# 汇总打印所有模型的分数
print("\n✅ 所有模型分数汇总:")
for k, v in all_scores.items():
    print(f"{k}: {v}")

In [ ]:
# ==================== 标准OOF: Out-Of-Fold验证集预测 ====================
# OOF 是一种评估和可视化方法: 使用交叉验证生成整个数据集的预测
# 步骤: 对于每个fold，用其他fold的数据训练，预测该fold的验证集
# 优势: 可以看到所有样本的预测（不仅是测试集），用于分析模型性能

# ========== 数据准备 ==========
# 使用fold_0的训练和验证数据
df = task.get_train_and_val_data(0, as_type="df")
df = df.rename(columns=lambda x: x.strip())

# 提取样本ID、目标值
mbids = np.array([str(x) for x in df.index], dtype=object)  # 材料项目ID
y = df[target_col].to_numpy(dtype=float)  # 真实的带隙值

# 加载之前提取的特征
all_features = pd.read_pickle(FEATURES_CACHE)
X = all_features["fold_0_train"]
if not isinstance(X, pd.DataFrame):
    X = pd.DataFrame(X)
# 缓存现存原始特征(可能含NaN)，填充在每个 OOF fold 内执行

# 验证特征和标签数量匹配
if len(X) != len(y):
    raise ValueError("特征和数据不匹配")

# ========== OOF参数设置 ==========
# RF参数配置
rf_params = {"n_estimators": 300, "random_state": SEED, "n_jobs": -1}

# XGBoost参数配置 (与之前的训练一致)
xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": "mae",
    "tree_method": "hist",
    "seed": SEED,
    "eta": 0.03,
    "max_depth": 6,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 1e-3
}

# ========== K折交叉验证生成OOF预测 ==========
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
# 初始化OOF预测字典（预先用NaN填充）
oof_dict = {model: np.full(len(y), np.nan) for model in MODELS_TO_RUN}

# 遍历每个fold
for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    print(f"\nOOF Fold {fold+1}/{N_SPLITS}")
    
    # 划分训练和验证集
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    # 填充: 只用本 OOF fold 训练子集的均值
    col_means = X_tr.mean(numeric_only=True)
    X_tr = X_tr.fillna(col_means).fillna(0).to_numpy()
    X_va = X_va.fillna(col_means).fillna(0).to_numpy()

    # ========== Random Forest OOF预测 ==========
    rf = RandomForestRegressor(**rf_params)
    rf.fit(X_tr, y_tr)  # 用训练集训练
    oof_dict["rf"][va_idx] = np.maximum(rf.predict(X_va), 0)  # 预测验证集 (clip)
    print(f"[RF] Fold MAE: {mean_absolute_error(y_va, oof_dict['rf'][va_idx]):.4f}")

    # ========== XGBoost OOF预测 ==========
    # XGBoost需要内部验证集来进行early stopping
    # 从训练集中再分出内部训练集和验证集
    inner_tr_idx, inner_va_idx = train_test_split(
        np.arange(len(X_tr)), test_size=0.1, random_state=SEED
    )
    
    # 创建XGBoost DMatrix对象
    dtr = xgb.DMatrix(X_tr[inner_tr_idx], label=y_tr[inner_tr_idx])  # 内部训练集
    dval = xgb.DMatrix(X_tr[inner_va_idx], label=y_tr[inner_va_idx])  # 内部验证集
    douter = xgb.DMatrix(X_va)  # 要预测的验证集
    
    # 训练XGBoost模型
    booster = xgb.train(
        xgb_params, dtr, 8000,
        evals=[(dval, "valid")],
        early_stopping_rounds=200,
        verbose_eval=False  # 不打印详细日志
    )
    
    # 获取最佳迭代次数，并用其预测
    best_it = booster.best_iteration
    try:
        oof_dict["xgb"][va_idx] = booster.predict(douter, iteration_range=(0, best_it + 1))
    except TypeError:
        oof_dict["xgb"][va_idx] = booster.predict(douter, ntree_limit=best_it + 1)
    oof_dict["xgb"][va_idx] = np.maximum(oof_dict["xgb"][va_idx], 0)  # clip
    print(f"[XGB] Fold MAE: {mean_absolute_error(y_va, oof_dict['xgb'][va_idx]):.4f} (best_it={best_it})")

# ========== 保存OOF预测结果 ==========
# 保存为.npz格式（压缩的numpy文件），便于后续分析
for model in MODELS_TO_RUN:
    np.savez(
        OUT_DIR / f"v1_oof_std_{model}.npz",
        mbid=mbids,  # 样本ID
        y_true=y,  # 真实值
        y_pred=oof_dict[model]  # 预测值
    )
    print(f"✅ {model} OOF预测已保存")

# ========== OOF可视化函数 ==========
def plot_oof(model_name, y_true, y_pred):
    """
    绘制OOF预测对比图
    
    参数:
        model_name: 模型名称
        y_true: 真实值
        y_pred: 预测值
    
    绘图内容:
    - 散点图: 真实值 vs 预测值
    - 对角线: 完美预测的参考线
    - 文本框: MAE, RMSE, R²分数
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.35, s=10)
    
    # 绘制 y=x 对角线（完美预测的情况）
    mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    plt.plot([mn, mx], [mn, mx], color='red', linestyle='--', label='Perfect Prediction')
    
    plt.xlabel("True Gap (eV)")
    plt.ylabel("Predicted Gap (eV)")
    plt.title(f"OOF Prediction vs True ({model_name})")
    
    # 添加性能指标文本框
    plt.text(
        0.05, 0.95,
        f"MAE={mae:.4f} eV\nRMSE={rmse:.4f} eV\nR²={r2:.4f}",
        transform=plt.gca().transAxes,
        va="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
    )
    
    plt.tight_layout()
    fig_path = OUT_DIR / f"v1_oof_pred_vs_true_{model_name}.png"
    plt.savefig(fig_path, dpi=200)
    plt.close()
    print(f"✅ OOF对比图保存: {fig_path}")

# 为每个模型绘制图表
for model in MODELS_TO_RUN:
    plot_oof(model, y, oof_dict[model])

# ========== 导出Top错误样本 ==========
def export_topk(model_name, mbids, y_true, y_pred, topk=TOPK):
    """
    导出误差最大的Top K个样本
    
    参数:
        model_name: 模型名称
        mbids: 样本ID数组
        y_true: 真实值
        y_pred: 预测值
        topk: 导出的样本数量
    
    返回:
        包含错误样本的DataFrame
    
    输出列:
    - mp_id: 材料项目ID
    - true_gap_eV: 真实带隙
    - pred_gap_eV: 预测带隙
    - abs_error_eV: 绝对误差
    """
    abs_err = np.abs(y_true - y_pred)  # 计算绝对误差
    idx = np.argsort(abs_err)[::-1][:topk]  # 取误差最大的topk个索引
    
    # 创建结果DataFrame
    df_top = pd.DataFrame({
        "mp_id": mbids[idx],
        "true_gap_eV": y_true[idx],
        "pred_gap_eV": y_pred[idx],
        "abs_error_eV": abs_err[idx]
    })
    
    # 保存为CSV文件
    out_csv = OUT_DIR / f"v1_abs_err_top{topk}_{model_name}.csv"
    df_top.to_csv(out_csv, index=False)
    print(f"✅ Top{topk} 错误样本保存: {out_csv}")
    return df_top

# 为每个模型导出Top错误样本
for model in MODELS_TO_RUN:
    export_topk(model, mbids, y, oof_dict[model])

In [ ]:
# ==================== 半导体候选材料榜单 ====================
# 基于RF和XGBoost双模型共识筛选，提高候选材料可信度

results_df = pd.DataFrame({
    "mp_id": mbids,
    "true_gap_eV": y,
    "pred_gap_rf_eV": oof_dict["rf"],
    "pred_gap_xgb_eV": oof_dict["xgb"],
    "pred_avg_eV": (oof_dict["rf"] + oof_dict["xgb"]) / 2,  # 双模型平均
    "abs_error_rf_eV": np.abs(y - oof_dict["rf"]),
    "abs_error_xgb_eV": np.abs(y - oof_dict["xgb"])
})

# 双模型共识筛选：两个模型都预测在0.5-3.0 eV范围内
min_good_gap, max_good_gap = 0.5, 3.0

good_semiconductors = results_df[
    (results_df['pred_gap_rf_eV'] >= min_good_gap) &
    (results_df['pred_gap_rf_eV'] <= max_good_gap) &
    (results_df['pred_gap_xgb_eV'] >= min_good_gap) &
    (results_df['pred_gap_xgb_eV'] <= max_good_gap)
].sort_values('pred_avg_eV', ascending=False)

print(f"\nTop 20 半导体候选材料 (双模型共识，预测带隙 {min_good_gap}~{max_good_gap} eV):")
print(good_semiconductors.head(20)[['mp_id', 'true_gap_eV', 'pred_gap_rf_eV', 'pred_gap_xgb_eV', 'pred_avg_eV']])

save_path = OUT_DIR / "candidates_gap_0.5_3.0eV_list.csv"
good_semiconductors.to_csv(save_path, index=False)
print(f"✅ 双模型共识候选名单保存: {save_path}")